# Week 63, Day 5: Class decorators

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w063d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def shout(cls):
    print("decorating", cls.__name__)
    cls.greeting = cls.greeting.upper()
    return cls

print("before")

@shout
class Hello:
    greeting = "hi"
    print("body runs")

print(Hello.greeting)

class Child(Hello):
    greeting = "hey"

print(Child.greeting)

In [ ]:
EXPORTERS = {}

def exporter(name):
    def decorate(cls):
        if not hasattr(cls, "export"):
            raise TypeError(f"{cls.__name__} needs an export method")
        EXPORTERS[name] = cls
        cls.format_name = name
        return cls
    return decorate

@exporter("csv")
class CsvExporter:
    def export(self, rows):
        return "\n".join(",".join(map(str, r)) for r in rows)

print(EXPORTERS["csv"] is CsvExporter, CsvExporter.format_name)
print(EXPORTERS["csv"]().export([[1, 2], [3, 4]]))
try:
    @exporter("broken")
    class Broken:
        pass
except TypeError as e:
    print("TypeError:", e)

In [ ]:
import functools

def logged(cls):
    def wrap(name, method):                      # a fresh scope for each method
        @functools.wraps(method)
        def wrapper(self, *args, **kwargs):
            print(f"-> {cls.__name__}.{name}{args}")
            return method(self, *args, **kwargs)
        return wrapper

    for name, value in list(vars(cls).items()):
        if callable(value) and not name.startswith("__"):
            setattr(cls, name, wrap(name, value))
    return cls

@logged
class Account:
    def __init__(self):
        self.balance = 0
    def deposit(self, amount):
        self.balance += amount
    def withdraw(self, amount):
        self.balance -= amount

acc = Account()
acc.deposit(50)
acc.withdraw(20)
print(acc.balance, Account.deposit.__name__)

@functools.total_ordering
class Version:
    def __init__(self, *parts):
        self.parts = parts
    def __eq__(self, other):
        return self.parts == other.parts
    def __lt__(self, other):
        return self.parts < other.parts

print(Version(1, 2) >= Version(1, 1), Version(2) <= Version(1, 9))

In [ ]:
import inspect

MISSING = object()

def simple_record(cls):
    names = list(inspect.get_annotations(cls))
    defaults = {n: vars(cls)[n] for n in names if n in vars(cls)}

    def __init__(self, *args, **kwargs):
        values = dict(zip(names, args))
        values.update(kwargs)
        for n in names:
            if n in values:
                setattr(self, n, values[n])
            elif n in defaults:
                setattr(self, n, defaults[n])
            else:
                raise TypeError(f"missing argument: {n}")

    def __repr__(self):
        inner = ", ".join(f"{n}={getattr(self, n)!r}" for n in names)
        return f"{cls.__name__}({inner})"

    cls.__init__, cls.__repr__ = __init__, __repr__
    return cls

@simple_record
class Book:
    title: str
    year: int = 2000

print(Book("Aké", 1981), Book(title="Ohio"), inspect.get_annotations(Book))

In [ ]:
def counted(cls):
    original_init = cls.__init__
    def __init__(self, *args, **kwargs):
        original_init(self, *args, **kwargs)
        cls.instances += 1
    cls.__init__ = __init__
    cls.instances = 0
    return cls                                   # the same class, changed in place

@counted
class Ticket:
    def __init__(self, seat):
        self.seat = seat

Ticket("A1"), Ticket("A2")
print(Ticket.instances, Ticket.__name__, type(Ticket("A3")).__name__, Ticket.instances)

> **Key idea.** A class decorator is a function called with the finished class (Cls = decorator(Cls)) after the metaclass and __init_subclass__ have run; with arguments it's a function returning the decorator. It can register, check, add methods with setattr, and wrap existing ones: loop over list(vars(cls).items()), make each wrapper in its own function call, use functools.wraps, and skip dunders and non-functions. inspect.get_annotations(cls) gives a class's own annotations in order, which is how dataclass-style generators find fields. Prefer changing the class in place and returning it over returning a new class. Pick the simplest tool: descriptors per attribute, decorators per class, __init_subclass__ per hierarchy, metaclasses only when nothing else will do.

### Exercise: Registering and adding

Write a decorator with an argument, `tagged(*tags)`, that sets `cls.tags` to a sorted tuple of the given tags **plus** any tags inherited from a base class's `tags`, records the class under **each** of its tags (inherited ones included) in the module-level dict `BY_TAG` (tag to a list of classes, in decoration order), and returns the class. Then a plain decorator `with_repr(cls)` that adds a `__repr__` showing the instance's attributes from `vars(self)` in insertion order, like `Point(x=1, y=2)`.

In [ ]:
BY_TAG = {}

def tagged(*tags):
    ___

def with_repr(cls):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"class decorators\"\n    @tagged(\"io\", \"slow\")\n    class Disk:\n        pass\n    @tagged(\"net\")\n    class Remote(Disk):\n        pass\n    assert Disk.tags == (\"io\", \"slow\") and Remote.tags == (\"io\", \"net\", \"slow\")\n    assert Disk in BY_TAG[\"io\"] and BY_TAG[\"net\"] == [Remote] and Remote in BY_TAG[\"slow\"]\n    @with_repr\n    class Point:\n        def __init__(self, x, y):\n            self.x, self.y = x, y\n    assert repr(Point(1, 2)) == \"Point(x=1, y=2)\"\n    @with_repr\n    @tagged(\"shape\")\n    class Circle:\n        def __init__(self):\n            self.r = 3\n    assert repr(Circle()) == \"Circle(r=3)\" and Circle.tags == (\"shape\",)")

### Exercise: Timing every method

Write `timed(clock)`: a decorator with an argument that wraps every **public** method defined in the class's own body (plain functions whose names don't start with `_`; leave everything else alone) so each call adds the time it took, measured with `clock()`, to `cls.timings[name]` (a dict made by the decorator, so each class has its own) and counts calls in `cls.calls[name]`. Keep each method's name and docstring with `functools.wraps`. Exceptions still propagate, but the call is still counted and timed.

In [ ]:
import functools

def timed(clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wrapall():\n    \"wrapping every method\"\n    ticks = iter(range(0, 1000, 5))\n    clock = lambda: next(ticks)\n    @timed(clock)\n    class Job:\n        def __init__(self):\n            self.done = []\n        def run(self, n):\n            \"Run n steps.\"\n            self.done.append(n)\n            return n * 2\n        def fail(self):\n            raise ValueError(\"no\")\n        def _helper(self):\n            return \"private\"\n        @staticmethod\n        def tool():\n            return \"static\"\n        limit = 3\n    j = Job()\n    assert j.run(2) == 4 and j.run(3) == 6\n    try:\n        j.fail()\n        assert False\n    except ValueError:\n        pass\n    assert Job.calls == {\"run\": 2, \"fail\": 1} and Job.timings == {\"run\": 10, \"fail\": 5}\n    assert Job.run.__name__ == \"run\" and Job.run.__doc__ == \"Run n steps.\"\n    assert j._helper() == \"private\" and Job.tool() == \"static\" and Job.limit == 3 and j.done == [2, 3]\n    @timed(clock)\n    class Other:\n        def go(self):\n            return 1\n    Other().go()\n    assert Other.calls == {\"go\": 1} and \"go\" not in Job.calls")

### Exercise: Ordered by fields

Write `ordered_by(*fields)`: a decorator that gives a class `__eq__`, `__hash__` and all four ordering comparisons (`<`, `<=`, `>`, `>=`), all based on the tuple of the named attributes. Comparing with an object of a different class returns `NotImplemented`. Don't use `functools.total_ordering`.

In [ ]:
def ordered_by(*fields):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ordering():\n    \"adding comparisons\"\n    @ordered_by(\"surname\", \"first\")\n    class Person:\n        def __init__(self, first, surname, age):\n            self.first, self.surname, self.age = first, surname, age\n    a, b, c = Person(\"Ada\", \"Lovelace\", 36), Person(\"Ada\", \"Byron\", 20), Person(\"Ada\", \"Lovelace\", 99)\n    assert a == c and a != b and b < a and a > b and a <= c and a >= c and not a < c\n    assert sorted([a, b]) == [b, a] and len({a, b, c}) == 2\n    assert (a == \"Ada\") is False\n    try:\n        a < 5\n        assert False\n    except TypeError:\n        pass\n    assert not uses(source, \"total_ordering\")")

### Exercise: Fields from annotations

Write `fields_of(cls)`: a list of `(name, default)` pairs for a class's annotated fields, including those of base classes (base fields first; a subclass annotation of the same name keeps the original position but takes the subclass's default), using `inspect.get_annotations` on each class in the MRO. A field without a default has the default `MISSING`. Then `check_order(cls)`, a decorator raising `TypeError` if a field without a default comes after one with a default (as `dataclass` does), and otherwise returning the class unchanged.

In [ ]:
import inspect

MISSING = object()

def fields_of(cls):
    ___

def check_order(cls):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fields():\n    \"reading annotations\"\n    class Base:\n        id: int\n        name: str = \"anon\"\n    class Child(Base):\n        active: bool = True\n        name: str = \"child\"\n    assert fields_of(Base) == [(\"id\", MISSING), (\"name\", \"anon\")]\n    assert fields_of(Child) == [(\"id\", MISSING), (\"name\", \"child\"), (\"active\", True)]\n    class Plain:\n        x = 1\n    assert fields_of(Plain) == []\n    @check_order\n    class Good(Base):\n        tags: list = None\n    assert Good.__name__ == \"Good\"\n    try:\n        @check_order\n        class Bad(Base):\n            required: int\n        assert False\n    except TypeError:\n        pass")

### Exercise: Frozen after __init__

Write a decorator `freeze_after_init(cls)`: instances can set attributes freely during `__init__`, but afterwards setting or deleting any attribute raises `AttributeError`. Wrap the class's `__init__` (keeping its behaviour) so it marks the instance as frozen when it finishes, and add `__setattr__` and `__delattr__` that check the mark. Subclasses whose own `__init__` calls `super().__init__()` and then sets more attributes must still work: only freeze when the **outermost** `__init__` call finishes (hint: compare `type(self)` with the decorated class, or count nesting).

In [ ]:
def freeze_after_init(cls):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_frozen():\n    \"a decorator that changes behaviour\"\n    @freeze_after_init\n    class Point:\n        def __init__(self, x, y):\n            self.x, self.y = x, y\n    p = Point(1, 2)\n    assert (p.x, p.y) == (1, 2)\n    for action in (lambda: setattr(p, \"x\", 5), lambda: delattr(p, \"y\"), lambda: setattr(p, \"z\", 0)):\n        try:\n            action()\n            assert False\n        except AttributeError:\n            pass\n    @freeze_after_init\n    class Point3(Point):\n        def __init__(self, x, y, z):\n            super().__init__(x, y)\n            self.z = z\n    q = Point3(1, 2, 3)\n    assert q.z == 3\n    try:\n        q.z = 4\n        assert False\n    except AttributeError:\n        pass\n    assert Point.__name__ == \"Point\"")

### Bug hunt: A decorator that lost the class

`stamp` sets the attribute but returns nothing, so `@stamp` replaces `Report` with `None` and the last line fails with `AttributeError`. Return the class so this prints `v2`.

In [ ]:
def stamp(cls):
    cls.version = "v2"

@stamp
class Report:
    pass

print(Report.version)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the class\"\n    expect_output(\"v2\")")

### Bug hunt: Every wrapper calls the last method

The wrappers are made in a loop, and each closure looks `method` up when it's **called**, by which time the loop has finished and `method` is the last one: this prints `b b`. Make each wrapper in its own function call so it prints `a b`.

In [ ]:
def quiet(cls):
    methods = [(n, m) for n, m in vars(cls).items() if callable(m) and not n.startswith("_")]
    for name, method in methods:
        def wrapper(self, *args):
            return method(self, *args)
        setattr(cls, name, wrapper)
    return cls

@quiet
class Letters:
    def a(self):
        return "a"
    def b(self):
        return "b"

x = Letters()
print(x.a(), x.b())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"late binding\"\n    expect_output(\"a b\")")

### Bug hunt: Changing the class while looping over it

The decorator adds a `get_` method for every attribute while looping directly over `vars(cls)`, which changes size during the loop, so this crashes with `RuntimeError`. Loop over a copy of the items so it prints `['get_colour', 'get_size']`.

In [ ]:
def getters(cls):
    for name, value in vars(cls).items():
        if not name.startswith("_"):
            setattr(cls, "get_" + name, lambda self, n=name: getattr(self, n))
    return cls

@getters
class Shirt:
    colour = "blue"
    size = "M"

print(sorted(n for n in vars(Shirt) if n.startswith("get_")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"loop over a copy\"\n    expect_output(\"['get_colour', 'get_size']\")")

### Puzzle: A decorator with an argument

Arrange the lines of a class decorator that takes a label and stores it on the class. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # @d above class C means: "C = d(C)" or "d runs before the body"
q = ___   # True or False: a class decorator on a base runs again for each subclass
r = ___   # which runs later: "__init_subclass__" or "the class decorator"
s = ___   # what to loop over when changing a class: "vars(cls).items()" or "list(vars(cls).items())"
t = ___   # True or False: functools.total_ordering is a class decorator

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"C = d(C)\" and q is False and r == \"the class decorator\", \"p, q and r\"\n    assert s == \"list(vars(cls).items())\" and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: your own dataclass

Write `record`, a class decorator usable as `@record` or with options `@record(order=True, frozen=True)` (both default `False`). It reads fields from the annotations of the class **and its bases** (base fields first, as in today's exercise) and:

- generates `__init__` taking the fields in order, positionally or by keyword, using defaults where given; a missing required argument raises `TypeError("missing field: x")`, an unknown keyword `TypeError("unknown field: x")`, and too many positional arguments `TypeError`;
- supports `Factory(func)` as a default: each instance gets a fresh `func()` (so `tags: list = Factory(list)` isn't shared);
- raises `TypeError` at decoration time if a field without a default follows one with a default;
- generates `__repr__` (`Point(x=1, y=2)`) and `__eq__` (same class and equal field values);
- with `order=True`, adds `<`, `<=`, `>`, `>=` comparing field tuples (same class only);
- with `frozen=True`, makes assignment and deletion raise `AttributeError` after `__init__`, and adds `__hash__` from the field values; without it, instances are unhashable (`__hash__ = None`);
- stores the field names as a tuple in `cls.__record_fields__` and returns the same class.

In [ ]:
import inspect

MISSING = object()

class Factory:
    def __init__(self, func):
        self.func = func

def record(cls=None, *, order=False, frozen=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basic():\n    \"init, repr, eq, defaults\"\n    @record\n    class Point:\n        x: int\n        y: int = 0\n    p = Point(1, 2)\n    assert p.x == 1 and repr(p) == \"Point(x=1, y=2)\" and p == Point(x=1, y=2) and Point(5) == Point(5, 0)\n    assert Point.__record_fields__ == (\"x\", \"y\") and p != Point(1, 3)\n    for args, kwargs, msg in (((), {}, \"missing field: x\"), ((1,), {\"z\": 2}, \"unknown field: z\")):\n        try:\n            Point(*args, **kwargs)\n            assert False\n        except TypeError as e:\n            assert str(e) == msg\n    try:\n        Point(1, 2, 3)\n        assert False\n    except TypeError:\n        pass\n    try:\n        hash(p)\n        assert False\n    except TypeError:\n        pass\n    p.x = 10\n    assert p.x == 10\n\ndef test_options():\n    \"factories, inheritance, order and frozen\"\n    @record\n    class Base:\n        id: int\n        tags: list = Factory(list)\n    @record(order=True, frozen=True)\n    class Item(Base):\n        price: int = 0\n    a, b = Item(1), Item(2, [\"x\"], 5)\n    assert a.tags == [] and a.tags is not Item(3).tags and b.price == 5\n    assert Item.__record_fields__ == (\"id\", \"tags\", \"price\") and repr(a) == \"Item(id=1, tags=[], price=0)\"\n    assert a < b and b >= a and sorted([b, a]) == [a, b]\n    try:\n        a.id = 9\n        assert False\n    except AttributeError:\n        pass\n    try:\n        del a.price\n        assert False\n    except AttributeError:\n        pass\n    c = Item(1, (), 0)\n    assert hash(c) == hash(Item(1, (), 0)) and len({c, Item(1, (), 0)}) == 1\n    try:\n        @record\n        class Bad:\n            a: int = 1\n            b: int\n        assert False\n    except TypeError:\n        pass\n    assert Base(1) != Item(1)")

## Warm-up

The class body runs first, then the decorator receives the finished class. And a decorator runs once, for the class it decorates: `Child` wasn't decorated, so its own `greeting` stayed lower case.

## A class decorator is a function

```
@decorator
class Item:
    ...
```

means exactly `Item = decorator(Item)`, run after the class has been created (after the metaclass, `__set_name__` and `__init_subclass__`: Thursday's step 6). The decorator can:

- **inspect** the class and **register** it somewhere, returning it unchanged;
- **check** it, raising an error at definition time if something is wrong;
- **modify** it in place, adding or replacing attributes with `setattr(cls, name, value)`, and return it;
- **return something else**, such as a subclass (rarely a good idea; see below).

A decorator **with arguments** is a function that returns the real decorator, as with functions in Week 60: `@register("csv")` calls `register("csv")` first and applies what it returns. Decorators stack, applied from the bottom up.

## Adding and wrapping methods

Functions are plain values, so a decorator can attach new methods with `setattr(cls, "__repr__", func)`, and they behave exactly like methods written in the body (Monday: functions are descriptors). `functools.total_ordering` works this way: given `__eq__` and one of `__lt__`, `__le__`, `__gt__`, `__ge__`, it adds the other three.

To **wrap** existing methods, loop over `vars(cls)` and replace each function with a wrapper. Three rules:

- Loop over a **copy**, `list(vars(cls).items())`, since you're changing the class as you go.
- Make each wrapper in a **separate function call** (a helper that takes the method), or every wrapper's closure ends up referring to the **last** method: the late-binding trap from Week 60.
- Use `functools.wraps` and skip what you shouldn't touch (dunders, `staticmethod` and `classmethod` objects, non-callables).

## Generating code from annotations

`@dataclass` (Week 32) reads the class's **annotations** (`x: int` in the body) and defaults (`y: int = 0`), then writes `__init__`, `__repr__` and `__eq__` for you. You can do the same:

- `inspect.get_annotations(cls)` returns the class's **own** annotations as a dict, in definition order. Use it rather than `cls.__annotations__`, whose behaviour has changed between Python versions (since 3.14, annotations are evaluated lazily).
- A field's default is the class attribute with that name, `vars(cls).get(name, MISSING)`.
- To include fields from base classes, walk `reversed(cls.__mro__)`.

The real `dataclass` builds the source code of each method as a string and runs it with `exec`, so the generated `__init__` has a proper signature and is fast. Closures, as below, are simpler and fine for learning.

## Returning a different class

A decorator may return a **new** class, typically a subclass that wraps the original:

```
def counted(cls):
    class Counted(cls):
        instances = 0
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            Counted.instances += 1
    return Counted
```

It works, but the name now refers to a different class: its `__name__` is `"Counted"` (unless you copy `__name__`, `__qualname__`, `__doc__` and `__module__` across), `super()` calls inside the original methods still refer to the original, and the original is hidden. Modifying the class in place and returning **the same class** avoids all of that, which is why almost every decorator in the standard library does so. (One exception: `dataclass(slots=True)` has to build a new class, because `__slots__` can't be added afterwards.)

## Choosing the right tool

This week gave you five ways to change how classes and attributes behave. In order from simplest to most powerful:

| You want | Use |
|---|---|
| behaviour for **one attribute**, reusable across classes | a **descriptor** (with `__set_name__`) or `property` |
| to handle **attributes that don't exist** | `__getattr__` |
| to change or register **one class**, explicitly | a **class decorator** |
| to run code for **every subclass** of a base, with arguments | **`__init_subclass__`** |
| to change the **namespace before** the class exists, control **calling** the class, or give classes **special methods** | a **metaclass** |

When two tools would work, prefer the one higher in the table: it's easier to read and combines better with other code. A decorator is visible right at the class; `__init_subclass__` is inherited invisibly but simply; a metaclass changes the type of every class in the hierarchy and can conflict with other metaclasses.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the source of `functools.total_ordering` (in `functools.py` in your Python installation, or online) and compare it with today's `ordered_by`. Tomorrow is the Forge: an object-mapping toolkit that uses descriptors, `__getattr__`, `__init_subclass__` and a class decorator together.